# PhysChemCAL — Kaggle Runner

Kaggle equivalent of `run_colab.ipynb` — same pipeline, but data comes from a Kaggle **Input** dataset instead of Google Drive (no Drive mount needed).

**Before running, in the Settings panel on the right:**
1. **Accelerator -> GPU** (T4 x2 or P100, whichever is offered).
2. **Internet -> On** (required for `git clone` and `pip install`).
3. **+ Add Input** -> attach the Kaggle Dataset containing your `.pkl` file. Once attached, its files are mounted read-only under `/kaggle/input/<your-dataset-slug>/...`.

Kaggle sessions have a time limit (12h) and a weekly GPU quota — same "commit results before you run out" discipline as Colab applies here too.

## 1. Check GPU

In [ ]:
!nvidia-smi

## 2. Configuration

Fill in `REPO_URL` once. `REPO_DIR` lives under `/kaggle/working/` — the only writable directory in a Kaggle notebook (`/kaggle/input/` is read-only, which is why the repo can't be cloned there).

In [ ]:
REPO_URL = "https://github.com/<your-username>/<your-repo>.git"  # TODO: fill in
REPO_DIR = "/kaggle/working/capstone_complete"
PKL_PATH = None  # left unset here -- Cell 3 auto-discovers it from /kaggle/input/

## 3. Auto-discover the `.pkl` under `/kaggle/input/`

Kaggle mounts attached datasets under `/kaggle/input/<dataset-slug>/...`, and the exact slug depends on how you named the dataset when uploading -- rather than guess it, this scans for any `.pkl` file under `/kaggle/input/` and sets `PKL_PATH` automatically if exactly one is found. If you have multiple `.pkl` files attached, it lists them so you can set `PKL_PATH` by hand instead.

In [ ]:
import glob

candidates = glob.glob("/kaggle/input/**/*.pkl", recursive=True)
print(f"Found {len(candidates)} .pkl file(s) under /kaggle/input/:")
for c in candidates:
    print(" ", c)

if len(candidates) == 1:
    PKL_PATH = candidates[0]
    print("\nAuto-selected PKL_PATH =", PKL_PATH)
elif len(candidates) == 0:
    raise FileNotFoundError(
        "No .pkl found under /kaggle/input/ -- check the dataset is attached via '+ Add Input' "
        "in the Settings panel, and that it actually contains a .pkl file."
    )
else:
    raise ValueError(
        "Multiple .pkl files found -- set PKL_PATH to the one you want explicitly, e.g.:\n"
        f"PKL_PATH = {candidates[0]!r}"
    )

## 4. Clone (or update) the repo into `/kaggle/working/`

In [ ]:
import os
if os.path.exists(REPO_DIR):
    %cd $REPO_DIR
    !git pull
else:
    !git clone "$REPO_URL" "$REPO_DIR"
    %cd $REPO_DIR

## 5. Install dependencies

Same reasoning as the Colab notebook: `torch` is left uninstalled -- Kaggle's base image already ships a torch build matched to its CUDA driver, and `requirements.txt` deliberately leaves it commented out to avoid clobbering that.

In [ ]:
!pip install -q -r requirements.txt

## 6. Sanity check: confirm the `.pkl` resolved correctly

In [ ]:
import os
assert PKL_PATH and os.path.exists(PKL_PATH), f"PKL_PATH ({PKL_PATH}) doesn't exist -- check Cell 3's output."
print("Using:", PKL_PATH, f"({os.path.getsize(PKL_PATH)/1e6:.1f} MB)")

## 7. Smoke test

Fast pipeline check on a truncated slice of the data. Run this first, especially the first time against a new `.pkl` -- it exercises the full pipeline without paying for a real training run.

Note: some DrugOOD entries are large peptides (500+ atoms); `--smoke-test` applies a built-in 150-atom cap automatically (see `--max-atoms` in the README) so this stays fast and doesn't OOM.

In [ ]:
!python main.py --pkl-path "$PKL_PATH" --smoke-test

## 8. Full training run (PhysChem + CAL only)

Trains for `--epochs` epochs, saves the best-validation-RMSE checkpoint to `checkpoints/best_model.pt`, and evaluates on the OOD test split at the end. Adjust `--epochs` / `--batch-size` / `--accumulation-steps` for your GPU, and reach for `--max-atoms` before shrinking `--batch-size` if you hit a CUDA OOM (see README).

In [ ]:
!python main.py --pkl-path "$PKL_PATH" --epochs 100 --batch-size 8

## 9. (Optional, expensive) Phase-3 counterfactual explanations

Only run this when you actually want counterfactuals. `--skip-train` reuses the checkpoint saved in step 8 instead of retraining. Pass `--query-smiles "<smiles>" ...` to explain specific molecules instead of randomly sampled OOD-test ones.

In [ ]:
!python main.py --pkl-path "$PKL_PATH" \
    --checkpoint checkpoints/best_model.pt --skip-train \
    --phase3 --n-queries 3

## 10. Commit results back to git

Every run appends its config + metrics to `results/<today>.md`. Kaggle sessions are ephemeral like Colab's, so push before the session ends rather than relying on it persisting.

(As a second safety net, everything under `/kaggle/working/` -- including `results/` and `checkpoints/` -- is also saved automatically as this notebook's Output when you commit/save the Kaggle notebook itself, even if you skip the git push below.)

In [ ]:
!git config --global user.email "you@example.com"   # TODO: fill in
!git config --global user.name "Your Name"           # TODO: fill in

In [ ]:
# Only needed if the repo is private: generate a GitHub Personal Access Token
# (github.com -> Settings -> Developer settings -> Fine-grained tokens) and paste it below.
# Leave blank if the repo is public or this session is already authenticated.
import getpass
token = getpass.getpass("GitHub token (leave blank if not needed): ")
if token:
    remote_url = REPO_URL.replace("https://", f"https://{token}@")
    !git remote set-url origin "$remote_url"

In [ ]:
!git add results/
!git commit -m "Kaggle run results: $(date +%Y-%m-%d)"
!git push